# UIT DSC 2026 LegalIR - Step 7 Qwen3 Reranker

Add `Qwen/Qwen3-Reranker-0.6B` on top of the best Step 6 candidate (`AITeamVN/Vietnamese_Reranker` fine-tuned). The notebook runs local inference only, tunes score fusion on dev, and writes public `submission.zip` files for public leaderboard checks.


In [ ]:
import sys, subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'sentence-transformers', 'transformers', 'accelerate', 'safetensors', 'sentencepiece'], check=True)


## Config And Input Contract


In [ ]:
from __future__ import annotations

import gc
import inspect
import json
import math
import os
import random
import re
import time
import zipfile
import unicodedata
from collections import Counter, defaultdict
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Iterable

import numpy as np

os.environ.setdefault('CUDA_VISIBLE_DEVICES', '0')
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

DATA_ROOT = Path('/kaggle/input/datasets/bowboochua9/stnhdscduaiti26')
OUTPUT_DIR = Path('/kaggle/working/step7')
PUBLIC_FILE = Path('/kaggle/input/datasets/ttdatto/uit-dsc26/LegalIR - Public Test/public-official.json')
MODEL_NAME = 'Qwen/Qwen3-Reranker-0.6B'
BEST_STEP6_SLUG = 'aiteamvn_vietnamese_reranker_finetuned'
MAX_SUBMISSION_DOCS = 5

ALLOWED_MODELS = {
    'BAAI/bge-m3',
    'bkai-foundation-models/vietnamese-bi-encoder',
    'AITeamVN/Vietnamese_Reranker',
    'itdainb/PhoRanker',
    'Qwen/Qwen3-Reranker-0.6B',
}
if MODEL_NAME not in ALLOWED_MODELS:
    raise ValueError(f'Model not whitelisted: {MODEL_NAME}')

# Path aliases only handle Kaggle unzip nesting for the same artifacts.
# They do not switch candidate pools or fall back to earlier step rankings.
def require_existing(name: str, candidates: list[Path]) -> Path:
    for candidate in candidates:
        if candidate.exists():
            return candidate
    checked = '\n'.join(str(p) for p in candidates)
    raise FileNotFoundError(f'Missing required Step 7 input: {name}. Checked:\n{checked}')

STEP4_ROOTS = [DATA_ROOT / 'step4', DATA_ROOT / 'step4' / 'step4']
STEP6_ROOTS = [DATA_ROOT / 'step7' / 'step6']
STEP6_BEST_ROOTS = [root / BEST_STEP6_SLUG for parent in STEP6_ROOTS for root in [parent / 'rankings']]
STEP6_CANDIDATE_ROOTS = [root / BEST_STEP6_SLUG for parent in STEP6_ROOTS for root in [parent / 'candidates']]
STEP6_REPORT_ROOTS = [root / 'reports' for root in STEP6_ROOTS]

CHUNKS_FILE = require_existing('step4/chunks.jsonl', [root / 'chunks.jsonl' for root in STEP4_ROOTS])
DEV_FILE = require_existing('step4/dev_split.json', [root / 'dev_split.json' for root in STEP4_ROOTS])
STEP6_DEV_FUSED = require_existing('step6 best dev fused rankings', [root / 'dev_rankings_step6_fused.jsonl' for root in STEP6_BEST_ROOTS])
STEP6_PUBLIC_FUSED = require_existing('step6 best public fused rankings', [root / 'public_rankings_step6_fused.jsonl' for root in STEP6_BEST_ROOTS])
STEP6_DEV_RERANKER = require_existing('step6 best dev reranker scores', [root / 'dev_rankings_reranker.jsonl' for root in STEP6_BEST_ROOTS])
STEP6_PUBLIC_RERANKER = require_existing('step6 best public reranker scores', [root / 'public_rankings_reranker.jsonl' for root in STEP6_BEST_ROOTS])
STEP6_RUN_REPORT = require_existing('step6/reports/run_report.json', [root / 'run_report.json' for root in STEP6_REPORT_ROOTS])
STEP6_PUBLIC_SCORES = require_existing('step6/reports/public_scores_manual.json', [root / 'public_scores_manual.json' for root in STEP6_REPORT_ROOTS])
STEP6_BEST_CONFIG = require_existing('step6 best fusion config', [root / 'best_fusion_config.json' for root in STEP6_CANDIDATE_ROOTS])

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for key, value in {
    'DATA_ROOT': DATA_ROOT,
    'CHUNKS_FILE': CHUNKS_FILE,
    'DEV_FILE': DEV_FILE,
    'PUBLIC_FILE': PUBLIC_FILE,
    'STEP6_DEV_FUSED': STEP6_DEV_FUSED,
    'STEP6_PUBLIC_FUSED': STEP6_PUBLIC_FUSED,
    'STEP6_DEV_RERANKER': STEP6_DEV_RERANKER,
    'STEP6_PUBLIC_RERANKER': STEP6_PUBLIC_RERANKER,
    'STEP6_RUN_REPORT': STEP6_RUN_REPORT,
    'STEP6_PUBLIC_SCORES': STEP6_PUBLIC_SCORES,
    'OUTPUT_DIR': OUTPUT_DIR,
}.items():
    print(f'{key}: {value}')


## Utilities


In [ ]:
def read_json(path: Path) -> Any:
    with path.open('r', encoding='utf-8') as f:
        return json.load(f)

def write_json(path: Path, payload: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w', encoding='utf-8') as f:
        json.dump(payload, f, ensure_ascii=False, indent=2, sort_keys=True)
        f.write('\n')

def iter_jsonl(path: Path) -> Iterable[dict[str, Any]]:
    with path.open('r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                yield json.loads(line)

def append_jsonl(path: Path, rows: Iterable[dict[str, Any]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w', encoding='utf-8') as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False, separators=(',', ':')) + '\n')

def make_submission(predictions: dict[str, list[str]]) -> dict[str, dict[str, list[str]]]:
    return {str(qid): {'answer': [str(doc_id) for doc_id in docs[:MAX_SUBMISSION_DOCS]]} for qid, docs in predictions.items()}

def validate_submission_payload(submission: dict[str, Any], public_payload: dict[str, Any], valid_doc_ids: set[str]) -> dict[str, Any]:
    issues = []
    expected_qids = set(map(str, public_payload.keys()))
    actual_qids = set(map(str, submission.keys()))
    for qid in sorted(expected_qids - actual_qids):
        issues.append({'level': 'error', 'query_id': qid, 'message': 'missing query_id'})
    for qid in sorted(actual_qids - expected_qids):
        issues.append({'level': 'error', 'query_id': qid, 'message': 'unexpected query_id'})
    lengths = Counter()
    for qid, row in submission.items():
        answer = row.get('answer') if isinstance(row, dict) else None
        if not isinstance(answer, list):
            issues.append({'level': 'error', 'query_id': str(qid), 'message': 'answer must be a list'})
            continue
        lengths[str(len(answer))] += 1
        if not (1 <= len(answer) <= MAX_SUBMISSION_DOCS):
            issues.append({'level': 'error', 'query_id': str(qid), 'message': 'answer length must be 1..5'})
        if len(answer) != len(set(answer)):
            issues.append({'level': 'error', 'query_id': str(qid), 'message': 'duplicate document_id'})
        for doc_id in answer:
            if not isinstance(doc_id, str):
                issues.append({'level': 'error', 'query_id': str(qid), 'message': 'document_id must be string'})
            elif doc_id not in valid_doc_ids:
                issues.append({'level': 'error', 'query_id': str(qid), 'message': f'unknown document_id: {doc_id}'})
    return {
        'num_public_queries': len(expected_qids),
        'num_submission_queries': len(actual_qids),
        'answer_length_distribution': dict(sorted(lengths.items())),
        'num_errors': sum(1 for issue in issues if issue['level'] == 'error'),
        'num_warnings': sum(1 for issue in issues if issue['level'] == 'warning'),
        'issues': issues[:200],
    }

def write_submission_zip(submission_json: Path, zip_path: Path) -> None:
    zip_path.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
        zf.write(submission_json, arcname='submission.json')

def evaluate_rankings(rankings: dict[str, list[str]], payload: dict[str, Any]) -> dict[str, Any]:
    rows = []
    for qid, row in payload.items():
        gold = [str(x) for x in row.get('answer', [])] if isinstance(row, dict) else []
        pred = [str(x) for x in rankings.get(str(qid), [])]
        top5 = pred[:MAX_SUBMISSION_DOCS]
        gold_set = set(gold)
        denom = len(gold_set) if gold_set else 1
        top5_hits = sum(1 for doc_id in top5 if doc_id in gold_set)
        hit_positions = [idx + 1 for idx, doc_id in enumerate(pred) if doc_id in gold_set]
        rows.append({
            'query_id': str(qid),
            'num_gold': len(gold_set),
            'exist@90': 1.0 if any(doc_id in gold_set for doc_id in pred[:90]) else 0.0,
            'hit@1': 1.0 if pred[:1] and pred[0] in gold_set else 0.0,
            'hit@5': 1.0 if top5_hits else 0.0,
            'hit@20': 1.0 if any(doc_id in gold_set for doc_id in pred[:20]) else 0.0,
            'mrr': 1.0 / hit_positions[0] if hit_positions else 0.0,
            'precision@5': top5_hits / MAX_SUBMISSION_DOCS,
            'recall@1': sum(1 for doc_id in pred[:1] if doc_id in gold_set) / denom,
            'recall@5': top5_hits / denom,
            'recall@20': sum(1 for doc_id in pred[:20] if doc_id in gold_set) / denom,
            'recall@50': sum(1 for doc_id in pred[:50] if doc_id in gold_set) / denom,
            'recall@90': sum(1 for doc_id in pred[:90] if doc_id in gold_set) / denom,
            'recall@100': sum(1 for doc_id in pred[:100] if doc_id in gold_set) / denom,
        })
    keys = [k for k in rows[0] if k not in {'query_id', 'num_gold'}] if rows else []
    return {'macro': {key: float(np.mean([row[key] for row in rows])) for key in keys}, 'per_query': rows}

def minmax(values: list[float]) -> list[float]:
    if not values:
        return []
    lo, hi = min(values), max(values)
    if abs(hi - lo) < 1e-12:
        return [0.5 for _ in values]
    return [(v - lo) / (hi - lo) for v in values]


## Load Corpus And Step 6 Rankings


In [ ]:
def load_chunks(chunks_file: Path) -> tuple[list[dict[str, Any]], dict[str, list[int]], set[str]]:
    chunks = []
    doc_to_chunk_indices: dict[str, list[int]] = defaultdict(list)
    valid_doc_ids = set()
    for idx, row in enumerate(iter_jsonl(chunks_file)):
        doc_id = str(row.get('doc_id', ''))
        chunk = {
            'chunk_idx': idx,
            'chunk_id': str(row.get('chunk_id', idx)),
            'doc_id': doc_id,
            'text': str(row.get('text', '')),
            'heading': str(row.get('heading', '')),
            'word_count': int(row.get('word_count') or 0),
            'metadata': row.get('metadata') if isinstance(row.get('metadata'), dict) else {},
        }
        chunks.append(chunk)
        if doc_id:
            doc_to_chunk_indices[doc_id].append(idx)
            valid_doc_ids.add(doc_id)
        if (idx + 1) % 50000 == 0:
            print(f'loaded {idx + 1:,} chunks')
    return chunks, doc_to_chunk_indices, valid_doc_ids

def load_fused_rankings(path: Path) -> dict[str, list[str]]:
    rankings = {}
    for row in iter_jsonl(path):
        qid = str(row.get('query_id'))
        if 'fused_doc_ids' not in row:
            raise ValueError(f'Missing fused_doc_ids in {path}: {row.keys()}')
        rankings[qid] = [str(x) for x in row['fused_doc_ids']]
    return rankings

def load_reranker_scores(path: Path) -> dict[str, dict[str, float]]:
    scores = {}
    for row in iter_jsonl(path):
        qid = str(row.get('query_id'))
        per_doc = {}
        for item in row.get('reranked_top_docs', []):
            per_doc[str(item['doc_id'])] = float(item['reranker_score'])
        scores[qid] = per_doc
    return scores

def strip_accents(text: str) -> str:
    text = unicodedata.normalize('NFD', text)
    text = ''.join(ch for ch in text if unicodedata.category(ch) != 'Mn')
    return unicodedata.normalize('NFC', text).replace('?', 'd').replace('?', 'D')

TOKEN_RE = re.compile(r'\b\w+\b', flags=re.UNICODE)
STOPWORDS = {'a','an','anh','ay','bi','boi','cac','can','cho','co','con','cua','duoc','da','de','den','di','do','doi','duoi','gi','hay','hoac','khi','la','lai','lam','mot','nay','neu','nhu','nhung','o','phai','qua','quy','rieng','sau','se','thi','theo','thuoc','toi','trong','tu','va','ve','vi','voi'}

def tokenize(text: str) -> list[str]:
    text = strip_accents(text.lower())
    return [tok for tok in TOKEN_RE.findall(text) if len(tok) > 1 and tok not in STOPWORDS]

def pick_evidence_text(question: str, doc_id: str, *, max_chunks: int = 3, max_chars: int = 2200) -> tuple[str, list[str]]:
    q = Counter(tokenize(question))
    q_set = set(q)
    scored = []
    for chunk_idx in doc_to_chunk_indices.get(str(doc_id), []):
        chunk = chunks[chunk_idx]
        toks = tokenize((chunk['heading'] + ' ' + chunk['text'])[:7000])
        cc = Counter(toks)
        overlap = sum(min(q[tok], cc[tok]) for tok in q_set)
        heading_bonus = 0.25 if any(tok in strip_accents(chunk['heading'].lower()) for tok in q_set) else 0.0
        score = overlap / max(1, len(q_set)) + heading_bonus
        scored.append((score, -abs(chunk['word_count'] - 320), chunk_idx))
    scored.sort(reverse=True)
    parts = []
    chunk_ids = []
    for _, _, chunk_idx in scored[:max_chunks]:
        chunk = chunks[chunk_idx]
        chunk_ids.append(chunk['chunk_id'])
        parts.append((chunk['heading'] + '\n' + chunk['text']).strip())
    text = '\n\n'.join(parts)
    if len(text) > max_chars:
        text = text[:max_chars]
    return text, chunk_ids

chunks, doc_to_chunk_indices, valid_doc_ids = load_chunks(CHUNKS_FILE)
dev_payload = read_json(DEV_FILE)
public_payload = read_json(PUBLIC_FILE)
step6_dev_rankings = load_fused_rankings(STEP6_DEV_FUSED)
step6_public_rankings = load_fused_rankings(STEP6_PUBLIC_FUSED)
step6_dev_scores = load_reranker_scores(STEP6_DEV_RERANKER)
step6_public_scores = load_reranker_scores(STEP6_PUBLIC_RERANKER)
step6_dev_metrics = evaluate_rankings(step6_dev_rankings, dev_payload)
write_json(OUTPUT_DIR / 'metrics' / 'step6_baseline_dev_metrics.json', step6_dev_metrics)
print(json.dumps({'chunks': len(chunks), 'docs': len(valid_doc_ids), 'dev_rankings': len(step6_dev_rankings), 'public_rankings': len(step6_public_rankings), 'step6_dev_recall@5': step6_dev_metrics['macro']['recall@5']}, ensure_ascii=False, indent=2))


## Qwen3 Rerank Top-50


In [ ]:
from sentence_transformers import CrossEncoder
from transformers.utils import logging as hf_logging
import torch

hf_logging.set_verbosity_error()

@dataclass(frozen=True)
class Step7Config:
    model_name: str = MODEL_NAME
    base_step6_slug: str = BEST_STEP6_SLUG
    rerank_top_docs: int = 50
    evidence_chunks: int = 3
    evidence_max_chars: int = 2200
    qwen_batch_size: int = 8
    max_length: int = 2048
    retrieval_weights: tuple[float, ...] = (0.1, 0.2, 0.3, 0.4, 0.5)
    step6_weights: tuple[float, ...] = (0.0, 0.2, 0.4, 0.6)
    qwen_weights: tuple[float, ...] = (0.3, 0.5, 0.7, 0.9)

config = Step7Config()
write_json(OUTPUT_DIR / 'configs' / 'step7_config.json', asdict(config))

def make_cross_encoder(model_name: str, *, max_length: int) -> CrossEncoder:
    device_name = 'cuda' if torch.cuda.is_available() else 'cpu'
    kwargs = {'max_length': max_length, 'device': device_name}
    signature = inspect.signature(CrossEncoder)
    if 'trust_remote_code' in signature.parameters:
        kwargs['trust_remote_code'] = True
    else:
        if 'model_kwargs' in signature.parameters:
            kwargs['model_kwargs'] = {'trust_remote_code': True}
        if 'tokenizer_kwargs' in signature.parameters:
            kwargs['tokenizer_kwargs'] = {'trust_remote_code': True}
    print({'cross_encoder_kwargs': kwargs})
    return CrossEncoder(model_name, **kwargs)

started = time.time()
model = make_cross_encoder(config.model_name, max_length=config.max_length)
param_count = int(sum(p.numel() for p in model.model.parameters()))
manifest = {
    'models': [
        {'model_id': 'BAAI/bge-m3', 'role': 'step4_dense_zero_shot_candidates_used_for_fusion', 'parameter_count': 568000000},
        {'model_id': 'bkai-foundation-models/vietnamese-bi-encoder', 'role': 'step5_fine_tuned_dense_biencoder', 'parameter_count': 134998272},
        {'model_id': 'AITeamVN/Vietnamese_Reranker', 'role': 'step6_fine_tuned_reranker_scores_used_for_fusion', 'parameter_count': 567755777},
        {'model_id': config.model_name, 'role': 'step7_qwen3_reranker_zero_shot', 'parameter_count': param_count},
    ],
    'allowed_models': sorted(ALLOWED_MODELS),
    'total_known_parameters': 568000000 + 134998272 + 567755777 + param_count,
    'max_total_parameters': 4000000000,
    'no_hosted_inference_or_api': True,
    'local_inference': True,
}
if manifest['total_known_parameters'] >= manifest['max_total_parameters']:
    raise ValueError('Parameter audit failed: total >= 4B')
write_json(OUTPUT_DIR / 'reports' / 'model_manifest.json', manifest)
print(json.dumps({'qwen_params': param_count, 'total_known_parameters': manifest['total_known_parameters']}, indent=2))


In [ ]:
def build_pair_rows(payload: dict[str, Any], base_rankings: dict[str, list[str]]) -> list[dict[str, Any]]:
    rows = []
    for qid, row in payload.items():
        qid = str(qid)
        question = row.get('question', '') if isinstance(row, dict) else ''
        for rank, doc_id in enumerate(base_rankings.get(qid, [])[:config.rerank_top_docs], start=1):
            evidence_text, chunk_ids = pick_evidence_text(question, doc_id, max_chunks=config.evidence_chunks, max_chars=config.evidence_max_chars)
            rows.append({'query_id': qid, 'question': question, 'doc_id': doc_id, 'text': evidence_text, 'retrieval_rank': rank, 'chunk_ids': chunk_ids})
    return rows

def score_pair_rows(pair_rows: list[dict[str, Any]], *, label: str) -> dict[str, list[dict[str, Any]]]:
    pairs = [(row['question'], row['text']) for row in pair_rows]
    raw_scores = model.predict(pairs, batch_size=config.qwen_batch_size, show_progress_bar=True)
    raw_scores = [float(x) for x in raw_scores]
    by_qid: dict[str, list[dict[str, Any]]] = defaultdict(list)
    for row, score in zip(pair_rows, raw_scores):
        by_qid[row['query_id']].append({
            'doc_id': row['doc_id'],
            'qwen_score': score,
            'retrieval_rank': row['retrieval_rank'],
            'chunk_ids': row['chunk_ids'],
        })
    for qid in by_qid:
        by_qid[qid].sort(key=lambda item: item['qwen_score'], reverse=True)
    print({'scored_pairs': len(pair_rows), 'label': label})
    return dict(by_qid)

def write_qwen_rankings(payload: dict[str, Any], base_rankings: dict[str, list[str]], qwen_scores: dict[str, list[dict[str, Any]]], output_file: Path) -> dict[str, list[str]]:
    rankings = {}
    def rows():
        for qid, row in payload.items():
            qid = str(qid)
            scored = qwen_scores.get(qid, [])
            reranked = [item['doc_id'] for item in scored]
            tail = [doc_id for doc_id in base_rankings.get(qid, []) if doc_id not in set(reranked)]
            doc_ids = reranked + tail
            rankings[qid] = doc_ids
            yield {'query_id': qid, 'question': row.get('question', '') if isinstance(row, dict) else '', 'gold': row.get('answer') if isinstance(row, dict) else None, 'qwen_top_docs': scored, 'qwen_doc_ids': doc_ids}
    append_jsonl(output_file, rows())
    return rankings

def qwen_score_map(qwen_scores: dict[str, list[dict[str, Any]]]) -> dict[str, dict[str, float]]:
    return {qid: {item['doc_id']: float(item['qwen_score']) for item in rows} for qid, rows in qwen_scores.items()}

def fuse_scores(base_rankings: dict[str, list[str]], step6_scores: dict[str, dict[str, float]], qwen_scores: dict[str, dict[str, float]], *, retrieval_weight: float, step6_weight: float, qwen_weight: float) -> dict[str, list[str]]:
    fused = {}
    for qid, base_docs in base_rankings.items():
        top_docs = base_docs[:config.rerank_top_docs]
        retrieval_raw = [1.0 / rank for rank in range(1, len(top_docs) + 1)]
        retrieval_norm = dict(zip(top_docs, minmax(retrieval_raw)))
        step6_raw = {doc_id: step6_scores.get(qid, {}).get(doc_id) for doc_id in top_docs if doc_id in step6_scores.get(qid, {})}
        qwen_raw = {doc_id: qwen_scores.get(qid, {}).get(doc_id) for doc_id in top_docs if doc_id in qwen_scores.get(qid, {})}
        step6_norm = dict(zip(step6_raw.keys(), minmax(list(step6_raw.values()))))
        qwen_norm = dict(zip(qwen_raw.keys(), minmax(list(qwen_raw.values()))))
        scores = {}
        first_seen = {}
        for idx, doc_id in enumerate(top_docs):
            first_seen.setdefault(doc_id, idx)
            scores[doc_id] = (
                retrieval_weight * retrieval_norm.get(doc_id, 0.0)
                + step6_weight * step6_norm.get(doc_id, 0.0)
                + qwen_weight * qwen_norm.get(doc_id, 0.0)
            )
        for idx, doc_id in enumerate(base_docs[config.rerank_top_docs:], start=len(top_docs)):
            first_seen.setdefault(doc_id, idx)
            scores.setdefault(doc_id, -idx * 1e-6)
        fused[qid] = [doc_id for doc_id, _ in sorted(scores.items(), key=lambda item: (-item[1], first_seen[item[0]]))[:100]]
    return fused


## Dev Ablation And Public Submissions


In [ ]:
dev_pairs = build_pair_rows(dev_payload, step6_dev_rankings)
dev_qwen_scores = score_pair_rows(dev_pairs, label='dev')
dev_qwen_only = write_qwen_rankings(dev_payload, step6_dev_rankings, dev_qwen_scores, OUTPUT_DIR / 'rankings' / 'dev_rankings_qwen3_only.jsonl')
dev_qwen_only_metrics = evaluate_rankings(dev_qwen_only, dev_payload)
write_json(OUTPUT_DIR / 'metrics' / 'dev_metrics_qwen3_only.json', dev_qwen_only_metrics)

dev_qwen_map = qwen_score_map(dev_qwen_scores)
trials = []
for retrieval_weight in config.retrieval_weights:
    for step6_weight in config.step6_weights:
        for qwen_weight in config.qwen_weights:
            if step6_weight == 0.0 and qwen_weight == 0.0:
                continue
            fused = fuse_scores(
                step6_dev_rankings,
                step6_dev_scores,
                dev_qwen_map,
                retrieval_weight=retrieval_weight,
                step6_weight=step6_weight,
                qwen_weight=qwen_weight,
            )
            metrics = evaluate_rankings(fused, dev_payload)['macro']
            trials.append({'retrieval_weight': retrieval_weight, 'step6_weight': step6_weight, 'qwen_weight': qwen_weight, 'metrics': metrics})
trials.sort(key=lambda r: (r['metrics']['recall@5'], r['metrics']['precision@5'], r['metrics']['recall@20'], r['metrics']['mrr']), reverse=True)
best_fusion = {k: trials[0][k] for k in ['retrieval_weight', 'step6_weight', 'qwen_weight']}
write_json(OUTPUT_DIR / 'metrics' / 'fusion_trials.json', trials)
write_json(OUTPUT_DIR / 'configs' / 'best_fusion_config.json', best_fusion)

dev_fused = fuse_scores(step6_dev_rankings, step6_dev_scores, dev_qwen_map, **best_fusion)
dev_fused_metrics = evaluate_rankings(dev_fused, dev_payload)
write_json(OUTPUT_DIR / 'metrics' / 'dev_metrics_step7_fused.json', dev_fused_metrics)

def metric_is_better(candidate: dict[str, float], baseline: dict[str, float], *, eps: float = 1e-12) -> bool:
    if candidate['recall@5'] > baseline['recall@5'] + eps:
        return True
    if abs(candidate['recall@5'] - baseline['recall@5']) <= eps and candidate['precision@5'] > baseline['precision@5'] + eps:
        return True
    return False

qwen_only_accepted = metric_is_better(dev_qwen_only_metrics['macro'], step6_dev_metrics['macro'])
step7_fused_accepted = metric_is_better(dev_fused_metrics['macro'], step6_dev_metrics['macro'])
step7_decision = {
    'baseline': 'step6_aiteamvn_vietnamese_reranker_finetuned',
    'qwen_only_accepted': qwen_only_accepted,
    'step7_fused_accepted': step7_fused_accepted,
    'accepted': qwen_only_accepted or step7_fused_accepted,
    'selection_rule': 'accept only if dev Recall@5 improves; Precision@5 is tie-break',
    'selected_public_slug': 'step7_fused_best' if step7_fused_accepted else ('qwen3_only' if qwen_only_accepted else 'step7_rejected_use_step6'),
    'reason': None,
}
if not step7_decision['accepted']:
    step7_decision['reason'] = 'Qwen3 reranker reduced dev Recall@5 versus Step 6, so public Qwen reranking is skipped.'
write_json(OUTPUT_DIR / 'reports' / 'dev_decision.json', step7_decision)

def final_dev_rows():
    for qid, row in dev_payload.items():
        qid = str(qid)
        yield {'query_id': qid, 'question': row.get('question', ''), 'gold': row.get('answer'), 'base_step6_doc_ids': step6_dev_rankings.get(qid, []), 'qwen_doc_ids': dev_qwen_only.get(qid, []), 'fused_doc_ids': dev_fused.get(qid, [])}
append_jsonl(OUTPUT_DIR / 'rankings' / 'dev_rankings_step7_fused.jsonl', final_dev_rows())
write_json(OUTPUT_DIR / 'predictions' / 'dev_predictions_top5_step7_fused.json', {qid: docs[:MAX_SUBMISSION_DOCS] for qid, docs in dev_fused.items()})

print('step6_baseline:', json.dumps(step6_dev_metrics['macro'], ensure_ascii=False, indent=2))
print('qwen_only:', json.dumps(dev_qwen_only_metrics['macro'], ensure_ascii=False, indent=2))
print('best_fusion:', best_fusion)
print('step7_fused:', json.dumps(dev_fused_metrics['macro'], ensure_ascii=False, indent=2))
print('step7_decision:', json.dumps(step7_decision, ensure_ascii=False, indent=2))


In [ ]:
def write_public_submission(slug: str, rankings: dict[str, list[str]]) -> dict[str, Any]:
    submission_dir = OUTPUT_DIR / 'submission' / slug
    submission = make_submission({qid: docs[:MAX_SUBMISSION_DOCS] for qid, docs in rankings.items()})
    validation = validate_submission_payload(submission, public_payload, valid_doc_ids)
    write_json(submission_dir / 'submission.json', submission)
    write_json(submission_dir / 'submission_validation.json', validation)
    if validation['num_errors']:
        raise ValueError(f"Submission validation failed for {slug}: {validation['num_errors']} errors")
    write_submission_zip(submission_dir / 'submission.json', submission_dir / 'submission.zip')
    return {'slug': slug, 'submission_zip': str(submission_dir / 'submission.zip'), 'submission_validation': str(submission_dir / 'submission_validation.json')}

public_outputs = []
public_qwen_only = None
public_fused = None

if not step7_decision['accepted']:
    public_fused = step6_public_rankings

    def rejected_public_rows():
        for qid, row in public_payload.items():
            qid = str(qid)
            yield {
                'query_id': qid,
                'question': row.get('question', ''),
                'gold': None,
                'base_step6_doc_ids': step6_public_rankings.get(qid, []),
                'qwen_doc_ids': None,
                'fused_doc_ids': step6_public_rankings.get(qid, []),
                'decision': step7_decision['reason'],
            }
    append_jsonl(OUTPUT_DIR / 'rankings' / 'public_rankings_step7_fused.jsonl', rejected_public_rows())
    public_outputs.append(write_public_submission('step7_rejected_use_step6', step6_public_rankings))
else:
    public_pairs = build_pair_rows(public_payload, step6_public_rankings)
    public_qwen_scores = score_pair_rows(public_pairs, label='public')
    public_qwen_only = write_qwen_rankings(public_payload, step6_public_rankings, public_qwen_scores, OUTPUT_DIR / 'rankings' / 'public_rankings_qwen3_only.jsonl')
    public_qwen_map = qwen_score_map(public_qwen_scores)
    public_fused = fuse_scores(step6_public_rankings, step6_public_scores, public_qwen_map, **best_fusion)

    def accepted_public_rows():
        for qid, row in public_payload.items():
            qid = str(qid)
            yield {'query_id': qid, 'question': row.get('question', ''), 'gold': None, 'base_step6_doc_ids': step6_public_rankings.get(qid, []), 'qwen_doc_ids': public_qwen_only.get(qid, []), 'fused_doc_ids': public_fused.get(qid, [])}
    append_jsonl(OUTPUT_DIR / 'rankings' / 'public_rankings_step7_fused.jsonl', accepted_public_rows())

    if step7_decision['qwen_only_accepted']:
        public_outputs.append(write_public_submission('qwen3_only', public_qwen_only))
    if step7_decision['step7_fused_accepted']:
        public_outputs.append(write_public_submission('step7_fused_best', public_fused))

write_json(OUTPUT_DIR / 'reports' / 'public_outputs.json', public_outputs)
print(json.dumps(public_outputs, ensure_ascii=False, indent=2))


## Report And Files


In [ ]:
report = {
    'inputs': {
        'chunks_file': str(CHUNKS_FILE),
        'dev_file': str(DEV_FILE),
        'public_file': str(PUBLIC_FILE),
        'step6_dev_fused': str(STEP6_DEV_FUSED),
        'step6_public_fused': str(STEP6_PUBLIC_FUSED),
        'step6_dev_reranker': str(STEP6_DEV_RERANKER),
        'step6_public_reranker': str(STEP6_PUBLIC_RERANKER),
        'step6_run_report': str(STEP6_RUN_REPORT),
        'step6_public_scores': str(STEP6_PUBLIC_SCORES),
    },
    'config': asdict(config),
    'model_manifest': manifest,
    'step6_baseline_dev_macro': step6_dev_metrics['macro'],
    'qwen_only_dev_macro': dev_qwen_only_metrics['macro'],
    'best_fusion': best_fusion,
    'step7_fused_dev_macro': dev_fused_metrics['macro'],
    'step7_decision': step7_decision,
    'public_outputs': public_outputs,
    'seconds': round(time.time() - started, 3),
    'source_notes': {
        'qwen_hf': 'https://huggingface.co/Qwen/Qwen3-Reranker-0.6B',
        'qwen_usage': 'Loaded locally through sentence-transformers CrossEncoder; no hosted inference/API.',
    },
}
write_json(OUTPUT_DIR / 'reports' / 'run_report.json', report)
print(json.dumps({
    'best_fusion': best_fusion,
    'step6_recall@5': step6_dev_metrics['macro']['recall@5'],
    'qwen_only_recall@5': dev_qwen_only_metrics['macro']['recall@5'],
    'step7_fused_recall@5': dev_fused_metrics['macro']['recall@5'],
    'step7_decision': step7_decision,
    'public_submission_zips': {row['slug']: row['submission_zip'] for row in public_outputs},
}, ensure_ascii=False, indent=2))


In [ ]:
for path in [
    OUTPUT_DIR / 'submission' / 'qwen3_only' / 'submission.zip',
    OUTPUT_DIR / 'submission' / 'step7_fused_best' / 'submission.zip',
    OUTPUT_DIR / 'submission' / 'qwen3_only' / 'submission_validation.json',
    OUTPUT_DIR / 'submission' / 'step7_fused_best' / 'submission_validation.json',
    OUTPUT_DIR / 'metrics' / 'step6_baseline_dev_metrics.json',
    OUTPUT_DIR / 'metrics' / 'dev_metrics_qwen3_only.json',
    OUTPUT_DIR / 'metrics' / 'dev_metrics_step7_fused.json',
    OUTPUT_DIR / 'metrics' / 'fusion_trials.json',
    OUTPUT_DIR / 'rankings' / 'dev_rankings_qwen3_only.jsonl',
    OUTPUT_DIR / 'rankings' / 'dev_rankings_step7_fused.jsonl',
    OUTPUT_DIR / 'rankings' / 'public_rankings_qwen3_only.jsonl',
    OUTPUT_DIR / 'rankings' / 'public_rankings_step7_fused.jsonl',
    OUTPUT_DIR / 'reports' / 'model_manifest.json',
    OUTPUT_DIR / 'reports' / 'run_report.json',
]:
    print(path, 'OK' if path.exists() else 'MISSING')
